In [14]:
"""
Grafo não direcionado com 8 vértices e 11 arestas (modelo próprio).
- BFS e DFS implementados com lista de adjacência
- Simulação de indisponibilidade de conexões
- Caminho antes/depois do bloqueio
"""
from collections import deque

# ---------- 1. MODELO ----------
# Lista de adjacência: dict[vértice] -> lista de vizinhos.
# Escolhi dict de listas porque o grafo é esparso (poucas arestas por
# vértice): ocupa O(V+E) de memória e percorrer vizinhos é direto.
ARESTAS = [
    ("cwb", "gru"), ("cwb", "gyn"), ("gru", "gyn"), ("gru", "cgb"), ("gyn", "cgb"),
    ("cgb", "bsb"), ("cgb", "bhz"), ("bsb", "bhz"),
    ("bhz", "gig"), ("bhz", "ssa"), ("gig", "ssa"),
]


class Grafo:
  def __init__(self, arestas):
    self.adj = {}
    self.bloqueadas = set()  # conexões indisponíveis (frozenset de 2 vértices)
    for a, b in arestas:
      self.adj.setdefault(a, []).append(b)
      self.adj.setdefault(b, []).append(a)

  # Em vez de apagar a aresta, marco como bloqueada: assim posso
  # bloquear e desbloquear sem reconstruir o grafo.
  def bloquear(self, a, b):
    self.bloqueadas.add(frozenset((a, b)))

  def desbloquear(self, a, b):
    self.bloqueadas.discard(frozenset((a, b)))

  def vizinhos(self, v):
    return [w for w in self.adj[v] if frozenset((v, w)) not in self.bloqueadas]

  # ---------- 2. BFS ----------
  # Fila (FIFO): explora por "camadas". Em grafo sem pesos, o primeiro
  # caminho que chega ao destino é o de MENOR número de conexões.
  def bfs(self, origem, destino):
    pai = {origem: None}
    fila = deque([origem])
    ordem = []
    while fila:
      v = fila.popleft()
      ordem.append(v)
      if v == destino:
        return self._caminho(pai, destino), ordem
      for w in self.vizinhos(v):
        if w not in pai:
          pai[w] = v
          fila.append(w)
    return None, ordem

  # ---------- 3. DFS ----------
  # Pilha (LIFO): vai fundo por um ramo antes de voltar. Acha UM caminho,
  # não necessariamente o mais curto. Versão iterativa (evita estourar
  # o limite de recursão do Python em grafos grandes).
  def dfs(self, origem, destino):
    pai = {origem: None}
    pilha = [origem]
    visitados = set()
    ordem = []
    while pilha:
      v = pilha.pop()
      if v in visitados:
        continue
      visitados.add(v)
      ordem.append(v)
      if v == destino:
        return self._caminho(pai, destino), ordem
      # reversed() para visitar vizinhos na ordem em que aparecem
      for w in reversed(self.vizinhos(v)):
        if w not in visitados:
          pai.setdefault(w, v)
          pilha.append(w)
    return None, ordem

  @staticmethod
  def _caminho(pai, destino):
    caminho = []
    v = destino
    while v is not None:
      caminho.append(v)
      v = pai[v]
    return caminho[::-1]


# ---------- 4. SIMULAÇÃO ----------
def txt(vertices):
  return " -> ".join(str(v) for v in vertices)


def mostrar(titulo, g, origem, destino):
  print(f"\n--- {titulo} ({origem} -> {destino}) ---")
  for nome, algoritmo in (("BFS", g.bfs), ("DFS", g.dfs)):
    caminho, ordem = algoritmo(origem, destino)
    if caminho:
      print(f"{nome}: {txt(caminho)}  [{len(caminho) - 1} conexões]")
    else:
      print(f"{nome}: nenhum caminho disponível")
    print(f"    ordem de visita: {', '.join(str(v) for v in ordem)}")


if __name__ == "__main__":
  g = Grafo(ARESTAS)
  print(f"Vértices: {len(g.adj)} | Arestas: {len(ARESTAS)}")

  origem, destino = "cwb", "ssa"

  # Cenário 1: uma conexão indisponível, com desvio possível
  mostrar("ANTES do bloqueio", g, origem, destino)
  g.bloquear("cgb", "bhz")
  print("\n>>> Conexão indisponível: cgb <-> bhz")
  mostrar("DEPOIS do bloqueio", g, origem, destino)

  # Cenário 2: duas conexões indisponíveis separam o grafo em duas partes
  g.bloquear("cgb", "bsb")
  print("\n>>> Conexões indisponíveis: cgb <-> bhz e cgb <-> bsb")
  mostrar("Cenário extremo (grafo desconectado)", g, origem, destino)


Vértices: 8 | Arestas: 11

--- ANTES do bloqueio (cwb -> ssa) ---
BFS: cwb -> gru -> cgb -> bhz -> ssa  [4 conexões]
    ordem de visita: cwb, gru, gyn, cgb, bsb, bhz, gig, ssa
DFS: cwb -> gru -> cgb -> bhz -> ssa  [4 conexões]
    ordem de visita: cwb, gru, gyn, cgb, bsb, bhz, gig, ssa

>>> Conexão indisponível: cgb <-> bhz

--- DEPOIS do bloqueio (cwb -> ssa) ---
BFS: cwb -> gru -> cgb -> bsb -> bhz -> ssa  [5 conexões]
    ordem de visita: cwb, gru, gyn, cgb, bsb, bhz, gig, ssa
DFS: cwb -> gru -> cgb -> bsb -> bhz -> ssa  [5 conexões]
    ordem de visita: cwb, gru, gyn, cgb, bsb, bhz, gig, ssa

>>> Conexões indisponíveis: cgb <-> bhz e cgb <-> bsb

--- Cenário extremo (grafo desconectado) (cwb -> ssa) ---
BFS: nenhum caminho disponível
    ordem de visita: cwb, gru, gyn, cgb
DFS: nenhum caminho disponível
    ordem de visita: cwb, gru, gyn, cgb
